# YOLO26s Detection Training Pipeline

**Pipeline overview:**
1. Reads COCO-format annotations from all dataset sources
2. Converts to YOLO detection format (normalised `.txt` labels)
3. Splits 50 full Self Images into test, rest into train/val
4. Filters bad annotations and handles class mapping
5. Creates `data.yaml` config
6. Trains `yolo26s.pt` detection model
7. Validates on test set with mAP metrics and saves a clean results summary

## 0 · Imports & Dependencies

In [1]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

try:
    import ultralytics
except ImportError:
    print("Installing ultralytics...")
    pip_install("ultralytics")

try:
    import yaml
except ImportError:
    pip_install("pyyaml")

In [2]:
import json
import os
import shutil
import random
import yaml
import datetime
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image

print("✅ All imports successful")

✅ All imports successful


## 1 · Configuration

Edit the cells below to match your setup before running the rest of the notebook.

In [3]:
# ── Paths ────────────────────────────────────────────────────────────
REPO_ROOT  = Path(".")          # Root folder containing 'Training Images/'
OUTPUT_DIR = Path("det_dataset") # Where the prepared YOLO dataset will be written

# ── Class definitions ────────────────────────────────────────────────
CLASS_NAMES = ["Bottled_Drink", "Canned", "Fresh_Produce", "Packaged_Food"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

# Known COCO category-ID → canonical class name
CATEGORY_MAP = {
    1: "Bottled_Drink",
    2: "Canned",
    3: "Fresh_Produce",
    4: "Packaged_Food",
}

# ── Split settings ───────────────────────────────────────────────────
NUM_TEST_SELF = 50   # Number of Self Images held out for the test split
VAL_FRACTION  = 0.15 # Fraction of training pool used for validation
SEED = 42
random.seed(SEED)

In [4]:
# ── Training hyperparameters ─────────────────────────────────────────
EPOCHS = 100
BATCH  = 8
IMGSZ  = 640

# ── Device selection ─────────────────────────────────────────────────
# Options:
#   "cpu"        → force CPU (slow)
#   "cuda"       → first CUDA GPU
#   "cuda:0"     → explicit GPU index
#   "0"          → shorthand for cuda:0
#   "0,1"        → multi-GPU (DDP)
#   "mps"        → Apple Silicon GPU
#   "auto"       → ultralytics picks the best available device
DEVICE = "auto"

# ── Results output ───────────────────────────────────────────────────
RUN_NAME    = "yolo26s_det_grocery"  # Sub-folder name inside RESULTS_DIR
RESULTS_DIR = Path("runs/detect")    # Parent folder for all training runs

In [5]:
# Quick sanity-check: show what device will be used
import torch

if DEVICE == "auto":
    if torch.cuda.is_available():
        _dev = f"CUDA ({torch.cuda.get_device_name(0)})"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        _dev = "MPS (Apple Silicon)"
    else:
        _dev = "CPU"
else:
    _dev = DEVICE

print(f"Device setting : {DEVICE!r}  →  will run on: {_dev}")
print(f"PyTorch version: {torch.__version__}")

Device setting : 'auto'  →  will run on: CUDA (NVIDIA GeForce RTX 3060 Laptop GPU)
PyTorch version: 2.11.0+cu130


## 2 · Helper Functions

In [6]:
# ── Dataset discovery ────────────────────────────────────────────────

DATASET_SPECS = [
    {
        "source": "kaggle",
        "ann": "Training Images/Kaggle Images/annotations.json",
        "img": "Training Images/Kaggle Images/images",
    },
    {
        "source": "mv",
        "ann": "Training Images/MV Images/selected_250.json",
        "img": "Training Images/MV Images/img",
    },
    {
        "source": "self_girvin",
        "ann": "Training Images/Self Images/annotated_resized_Girvin-images/train/_annotations.coco.json",
        "img": "Training Images/Self Images/annotated_resized_Girvin-images/train",
    },
    {
        "source": "self_naveen",
        "ann": "Training Images/Self Images/annotated_resized_Naveen-images/train/_annotations.coco.json",
        "img": "Training Images/Self Images/annotated_resized_Naveen-images/train",
    },
    {
        "source": "self_stanley_add",
        "ann": "Training Images/Self Images/stanley_add_dataset/coco_annotations.json",
        "img": "Training Images/Self Images/stanley_add_dataset",
    },
    {
        "source": "self_stanley_orig",
        "ann": "Training Images/Self Images/stanley_dataset/stanley_original/annotations.json",
        "img": "Training Images/Self Images/stanley_dataset/stanley_original/images",
    },
]


def find_coco_datasets(repo_root: Path):
    """Walk the repo and return list of dataset dicts that actually exist."""
    datasets = []
    for spec in DATASET_SPECS:
        ann_path = repo_root / spec["ann"]
        img_path = repo_root / spec["img"]
        if ann_path.exists():
            datasets.append({"ann_file": ann_path, "img_dir": img_path, "source": spec["source"]})
            print(f"  [✓] {spec['source']}")
        else:
            print(f"  [✗] {spec['source']} — not found at {ann_path}")
    return datasets

In [7]:
# ── Annotation parsing ───────────────────────────────────────────────

def map_category(cat_id, cat_name):
    """Map a COCO category to our canonical class index (returns None if unknown)."""
    if cat_id in CATEGORY_MAP:
        return CLASS_TO_IDX.get(CATEGORY_MAP[cat_id])
    name_lower = cat_name.lower().replace(" ", "_")
    for canon_name, idx in CLASS_TO_IDX.items():
        if canon_name.lower() in name_lower or name_lower in canon_name.lower():
            return idx
    return None


def parse_coco_dataset(ann_file: Path, img_dir: Path, source: str):
    """
    Parse a COCO JSON and return a list of image records:
    [{ img_path, width, height, labels: [(cls, xc, yc, w, h)], source, img_id }]
    All bbox values are normalised to [0, 1].
    """
    with open(ann_file) as f:
        coco = json.load(f)

    id_to_info = {img["id"]: img for img in coco.get("images", [])}

    cat_id_to_cls = {}
    for cat in coco.get("categories", []):
        cls_idx = map_category(cat["id"], cat["name"])
        if cls_idx is not None:
            cat_id_to_cls[cat["id"]] = cls_idx

    anns_by_img = defaultdict(list)
    skipped = 0

    for ann in coco.get("annotations", []):
        cat_id = ann["category_id"]
        if cat_id not in cat_id_to_cls:
            skipped += 1
            continue

        cls_idx  = cat_id_to_cls[cat_id]
        bbox_raw = ann.get("bbox")

        # Fall back to segmentation polygon if no bbox
        if bbox_raw is None or bbox_raw == []:
            seg = ann.get("segmentation", [])
            if seg and isinstance(seg, list) and len(seg) > 0:
                pts = seg[0] if isinstance(seg[0], list) else seg
                try:
                    xs = [float(pts[i]) for i in range(0, len(pts), 2)]
                    ys = [float(pts[i]) for i in range(1, len(pts), 2)]
                    bbox_raw = [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]
                except (ValueError, IndexError):
                    skipped += 1
                    continue
            else:
                skipped += 1
                continue

        try:
            if isinstance(bbox_raw, list) and len(bbox_raw) == 1 and isinstance(bbox_raw[0], list):
                bbox_raw = bbox_raw[0]
            x, y, w, h = [float(v) for v in bbox_raw]
        except (ValueError, TypeError):
            skipped += 1
            continue

        if w <= 0 or h <= 0:
            skipped += 1
            continue

        anns_by_img[ann["image_id"]].append((cls_idx, x, y, w, h))

    records = []
    missing = 0

    for img_id, labels in anns_by_img.items():
        info = id_to_info.get(img_id)
        if info is None:
            missing += 1
            continue

        img_path = img_dir / info["file_name"]
        if not img_path.exists():
            alt = img_dir.parent / info["file_name"]
            if alt.exists():
                img_path = alt
            else:
                missing += 1
                continue

        img_w = info.get("width")
        img_h = info.get("height")
        if not img_w or not img_h:
            try:
                with Image.open(img_path) as im:
                    img_w, img_h = im.size
            except Exception:
                missing += 1
                continue

        yolo_labels = []
        for cls_idx, bx, by, bw, bh in labels:
            xc = max(0.0, min(1.0, (bx + bw / 2) / img_w))
            yc = max(0.0, min(1.0, (by + bh / 2) / img_h))
            nw = max(0.001, min(1.0, bw / img_w))
            nh = max(0.001, min(1.0, bh / img_h))
            if nw < 0.005 and nh < 0.005:  # skip sub-pixel boxes
                continue
            yolo_labels.append((cls_idx, xc, yc, nw, nh))

        if yolo_labels:
            records.append({
                "img_path": img_path, "width": img_w, "height": img_h,
                "labels": yolo_labels, "source": source, "img_id": img_id,
            })

    if skipped:
        print(f"    ⚠  {skipped} annotations skipped (bad bbox / unknown category)")
    if missing:
        print(f"    ⚠  {missing} images not found on disk")

    return records

In [8]:
# ── Dataset writing ──────────────────────────────────────────────────

def write_yolo_split(records, split_name, output_dir):
    img_out = output_dir / "images" / split_name
    lbl_out = output_dir / "labels" / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for i, rec in enumerate(records):
        suffix   = rec["img_path"].suffix
        new_name = f"{rec['source']}_{rec['img_id']}_{i:05d}{suffix}"
        shutil.copy2(rec["img_path"], img_out / new_name)
        lbl_path = lbl_out / (Path(new_name).stem + ".txt")
        with open(lbl_path, "w") as f:
            for cls_idx, xc, yc, w, h in rec["labels"]:
                f.write(f"{cls_idx} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

    return len(records)


def write_data_yaml(output_dir: Path):
    data = {
        "path":  str(output_dir.resolve()),
        "train": "images/train",
        "val":   "images/val",
        "test":  "images/test",
        "nc":    len(CLASS_NAMES),
        "names": CLASS_NAMES,
    }
    yaml_path = output_dir / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data, f, default_flow_style=False)
    return yaml_path

## 3 · Prepare Dataset

In [9]:
print("Discovering datasets...")
datasets = find_coco_datasets(REPO_ROOT)

if not datasets:
    raise RuntimeError("❌ No datasets found. Check REPO_ROOT.")

print(f"\n{len(datasets)} source(s) found.")

Discovering datasets...
  [✓] kaggle
  [✓] mv
  [✓] self_girvin
  [✓] self_naveen
  [✗] self_stanley_add — not found at Training Images/Self Images/stanley_add_dataset/coco_annotations.json
  [✓] self_stanley_orig

5 source(s) found.


In [10]:
print("Parsing COCO annotations → YOLO format...\n")

all_records  = []
self_records = []

for ds in datasets:
    print(f"  {ds['source']}")
    records = parse_coco_dataset(ds["ann_file"], ds["img_dir"], ds["source"])
    n_boxes = sum(len(r["labels"]) for r in records)
    print(f"    → {len(records)} images, {n_boxes} boxes")
    all_records.extend(records)
    if ds["source"].startswith("self_"):
        self_records.extend(records)

total_boxes = sum(len(r["labels"]) for r in all_records)
print(f"\nTotal  : {len(all_records)} images, {total_boxes} boxes")
print(f"Self   : {len(self_records)} images")

class_counts = Counter()
for r in all_records:
    for cls_idx, *_ in r["labels"]:
        class_counts[CLASS_NAMES[cls_idx]] += 1

print("\nOverall class distribution:")
for cls in CLASS_NAMES:
    pct = 100 * class_counts[cls] / total_boxes if total_boxes else 0
    print(f"  {cls:<20} {class_counts[cls]:>6}  ({pct:.1f}%)")

Parsing COCO annotations → YOLO format...

  kaggle
    → 200 images, 2078 boxes
  mv
    → 250 images, 1290 boxes
  self_girvin
    → 21 images, 53 boxes
  self_naveen
    → 35 images, 85 boxes
  self_stanley_orig
    → 32 images, 62 boxes

Total  : 538 images, 3568 boxes
Self   : 88 images

Overall class distribution:
  Bottled_Drink          1410  (39.5%)
  Canned                 1414  (39.6%)
  Fresh_Produce           526  (14.7%)
  Packaged_Food           218  (6.1%)


In [11]:
print("Splitting dataset...\n")

random.shuffle(self_records)
n_test       = min(NUM_TEST_SELF, len(self_records))
test_records = self_records[:n_test]
remaining    = self_records[n_test:]

non_self   = [r for r in all_records if not r["source"].startswith("self_")]
train_pool = non_self + remaining
random.shuffle(train_pool)

val_size     = max(1, int(len(train_pool) * VAL_FRACTION))
val_records  = train_pool[:val_size]
train_records = train_pool[val_size:]

splits = {"train": train_records, "val": val_records, "test": test_records}

print(f"  {'Split':<8}  {'Images':>7}  {'Boxes':>7}")
print(f"  {'─'*8}  {'─'*7}  {'─'*7}")
for split, recs in splits.items():
    boxes = sum(len(r["labels"]) for r in recs)
    src   = "(Self only)" if split == "test" else ""
    print(f"  {split:<8}  {len(recs):>7}  {boxes:>7}  {src}")

Splitting dataset...

  Split      Images    Boxes
  ────────  ───────  ───────
  train         415     2903  
  val            73      546  
  test           50      119  (Self only)


In [12]:
print("Writing YOLO dataset to disk...")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for split_name, recs in splits.items():
    n = write_yolo_split(recs, split_name, OUTPUT_DIR)
    print(f"  {split_name}: {n} images written")

yaml_path = write_data_yaml(OUTPUT_DIR)
print(f"\n✅ Dataset ready at: {OUTPUT_DIR.resolve()}")
print(f"   data.yaml      : {yaml_path}")

Writing YOLO dataset to disk...
  train: 415 images written
  val: 73 images written
  test: 50 images written

✅ Dataset ready at: /var/home/polygon/EE3703_RPC/det_dataset
   data.yaml      : det_dataset/data.yaml


In [13]:
# Per-split class distribution table
print(f"\n{'Class':<22}", end="")
for split in splits:
    print(f"  {split:>10}", end="")
print()
print("─" * (22 + 12 * len(splits)))

for cls in CLASS_NAMES:
    print(f"{cls:<22}", end="")
    for recs in splits.values():
        cnt = sum(1 for r in recs for ci, *_ in r["labels"] if CLASS_NAMES[ci] == cls)
        print(f"  {cnt:>10}", end="")
    print()


Class                        train         val        test
──────────────────────────────────────────────────────────
Bottled_Drink                 1178         209          23
Canned                        1173         223          18
Fresh_Produce                  372          95          59
Packaged_Food                  180          19          19


## 4 · Train Model

In [14]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")
print(f"Loaded model: yolo26s.pt")
print(f"Device      : {DEVICE!r}")
print(f"Epochs      : {EPOCHS}")
print(f"Batch       : {BATCH}")
print(f"Image size  : {IMGSZ}")

Loaded model: yolo26s.pt
Device      : 'auto'
Epochs      : 100
Batch       : 8
Image size  : 640


In [15]:
results = model.train(
    data    = str(yaml_path.resolve()),
    epochs  = EPOCHS,
    imgsz   = IMGSZ,
    batch   = BATCH,
    device  = DEVICE,
    # Early stopping
    patience = 20,
    # Optimiser
    optimizer      = "MuSGD",
    lr0            = 0.01,
    lrf            = 0.001,
    weight_decay   = 0.0005,
    warmup_epochs  = 5,
    warmup_momentum = 0.8,
    # Augmentation
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    degrees    = 10.0,
    translate  = 0.1,
    scale      = 0.5,
    fliplr     = 0.5,
    flipud     = 0.0,
    mosaic     = 1.0,
    mixup      = 0.1,
    copy_paste = 0.1,
    erasing    = 0.1,
    # Output location
    name     = RUN_NAME,
    exist_ok = True,
    verbose  = True,
)

save_dir = Path(results.save_dir)
best_pt  = save_dir / "weights" / "best.pt"
print(f"\n✅ Training complete!")
print(f"   Run directory : {save_dir}")
print(f"   Best weights  : {best_pt}")

Ultralytics 8.4.31 🚀 Python-3.11.15 torch-2.11.0+cu130 CUDA:auto (NVIDIA GeForce RTX 3060 Laptop GPU, 5781MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/var/home/polygon/EE3703_RPC/det_dataset/data.yaml, degrees=10.0, deterministic=True, device=auto, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.1, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26s_det_grocery, nbs=64, nms=False, opset=None, optimize=Fal

## 5 · Evaluate on Test Set

In [16]:
best_model   = YOLO(str(best_pt))
test_metrics = best_model.val(
    data    = str(yaml_path.resolve()),
    split   = "test",
    imgsz   = IMGSZ,
    device  = DEVICE,
    name     = f"{RUN_NAME}/test_eval",
    exist_ok = True,
)

print("✅ Evaluation complete")

Ultralytics 8.4.31 🚀 Python-3.11.15 torch-2.11.0+cu130 CUDA:auto (NVIDIA GeForce RTX 3060 Laptop GPU, 5781MiB)
YOLO26s summary (fused): 122 layers, 9,466,728 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 484.9±326.1 MB/s, size: 132.0 KB)
val: Scanning /var/home/polygon/EE3703_RPC/det_dataset/labels/test... 50 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 3.7Kit/s 0.0s
val: New cache created: /var/home/polygon/EE3703_RPC/det_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.8it/s 0.8s0.3s
                   all         50        119      0.914      0.918      0.953      0.851
         Bottled_Drink         23         23      0.914      0.925      0.953      0.874
                Canned         16         18      0.888          1      0.987      0.881
         Fresh_Produce         31         59       0.93      0.907      0.956      0.832
        

In [17]:
# ── Pretty-print results ─────────────────────────────────────────────
box = test_metrics.box

header = "Test Set Results"
print(f"\n{'─'*40}")
print(f"  {header}")
print(f"{'─'*40}")
print(f"  {'mAP@0.50':<18}  {box.map50:.4f}")
print(f"  {'mAP@0.50:0.95':<18}  {box.map:.4f}")
print(f"  {'Precision':<18}  {box.mp:.4f}")
print(f"  {'Recall':<18}  {box.mr:.4f}")
print(f"{'─'*40}")
print(f"  Per-class mAP@0.50")
for i, cls_name in enumerate(CLASS_NAMES):
    if i < len(box.ap50):
        print(f"    {cls_name:<20}  {box.ap50[i]:.4f}")
print(f"{'─'*40}")


────────────────────────────────────────
  Test Set Results
────────────────────────────────────────
  mAP@0.50            0.9533
  mAP@0.50:0.95       0.8515
  Precision           0.9144
  Recall              0.9184
────────────────────────────────────────
  Per-class mAP@0.50
    Bottled_Drink         0.9529
    Canned                0.9867
    Fresh_Produce         0.9563
    Packaged_Food         0.9174
────────────────────────────────────────


## 5b · Inference Speed Benchmark

In [18]:
# ── Inference Speed Benchmark ─────────────────────────────────────────────────
# Measures real-world latency on the test set images.
# Reports:
#   • Per-image mean / std / min / max latency  (ms)
#   • Throughput  (FPS)
#   • Ultralytics internal breakdown (preprocess / inference / postprocess)
# ─────────────────────────────────────────────────────────────────────────────

import time, statistics, glob
import torch
from pathlib import Path

# ── Settings ─────────────────────────────────────────────────────────────────
N_WARMUP   = 10   # warm-up images (discarded, lets CUDA reach steady state)
BENCH_SPLIT = "test"   # which split to benchmark on

# ── Collect image paths ───────────────────────────────────────────────────────
bench_images = sorted(
    (OUTPUT_DIR / "images" / BENCH_SPLIT).glob("*.[jp][pn][eg]*")
)
if not bench_images:
    raise FileNotFoundError(f"No images found in {OUTPUT_DIR}/images/{BENCH_SPLIT}")

print(f"Benchmarking on {len(bench_images)} images  "
      f"(imgsz={IMGSZ}, device={DEVICE})")
print(f"Warm-up: {N_WARMUP} images  |  Timed: {len(bench_images)} images")
print()

# Ensure CUDA streams are synchronised before timing
def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# ── Warm-up pass ──────────────────────────────────────────────────────────────
print("⏳ Warming up ...")
for img_path in bench_images[:N_WARMUP]:
    best_model.predict(str(img_path), imgsz=IMGSZ, device=DEVICE,
                       verbose=False, save=False)
sync()
print("   Warm-up done.")
print()

# ── Timed pass ────────────────────────────────────────────────────────────────
latencies_ms = []   # wall-clock time per image (preprocess + infer + post)
pre_ms_list, inf_ms_list, post_ms_list = [], [], []

for img_path in bench_images:
    sync()
    t0 = time.perf_counter()
    result = best_model.predict(str(img_path), imgsz=IMGSZ, device=DEVICE,
                                verbose=False, save=False)
    sync()
    t1 = time.perf_counter()

    latencies_ms.append((t1 - t0) * 1_000)

    # Ultralytics stores per-image speed in result[0].speed  (ms)
    spd = result[0].speed          # {'preprocess': x, 'inference': y, 'postprocess': z}
    pre_ms_list.append(spd.get("preprocess", 0.0))
    inf_ms_list.append(spd.get("inference",  0.0))
    post_ms_list.append(spd.get("postprocess", 0.0))

# ── Compute statistics ────────────────────────────────────────────────────────
def stats(lst):
    return {
        "mean": statistics.mean(lst),
        "std":  statistics.stdev(lst) if len(lst) > 1 else 0.0,
        "min":  min(lst),
        "max":  max(lst),
    }

wall  = stats(latencies_ms)
pre   = stats(pre_ms_list)
inf   = stats(inf_ms_list)
post  = stats(post_ms_list)

fps = 1_000.0 / wall["mean"]

# ── Pretty print ──────────────────────────────────────────────────────────────
print("─" * 52)
print("  Inference Speed Benchmark Results")
print("─" * 52)
print(f"  Images timed          : {len(latencies_ms)}")
print(f"  Image size            : {IMGSZ} × {IMGSZ}")
print()
print(f"  Wall-clock latency (ms/image)")
print(f"    Mean  : {wall['mean']:7.2f}")
print(f"    Std   : {wall['std']:7.2f}")
print(f"    Min   : {wall['min']:7.2f}")
print(f"    Max   : {wall['max']:7.2f}")
print()
print(f"  Throughput            : {fps:.1f} FPS")
print()
print(f"  Ultralytics breakdown (ms/image, mean)")
print(f"    Preprocess          : {pre['mean']:7.2f}")
print(f"    Inference           : {inf['mean']:7.2f}")
print(f"    Postprocess         : {post['mean']:7.2f}")
print(f"    ─────────────────────────────")
print(f"    Total (framework)   : {pre['mean']+inf['mean']+post['mean']:7.2f}")
print("─" * 52)

# ── Store for summary ─────────────────────────────────────────────────────────
speed_results = {
    "bench_images":          len(latencies_ms),
    "latency_mean_ms":       round(wall["mean"], 3),
    "latency_std_ms":        round(wall["std"],  3),
    "latency_min_ms":        round(wall["min"],  3),
    "latency_max_ms":        round(wall["max"],  3),
    "throughput_fps":        round(fps, 2),
    "preprocess_mean_ms":    round(pre["mean"],  3),
    "inference_mean_ms":     round(inf["mean"],  3),
    "postprocess_mean_ms":   round(post["mean"], 3),
}


Benchmarking on 27 images  (imgsz=640, device=auto)
Warm-up: 10 images  |  Timed: 27 images

⏳ Warming up ...
   Warm-up done.

────────────────────────────────────────────────────
  Inference Speed Benchmark Results
────────────────────────────────────────────────────
  Images timed          : 27
  Image size            : 640 × 640

  Wall-clock latency (ms/image)
    Mean  :    8.75
    Std   :    0.39
    Min   :    8.18
    Max   :    9.81

  Throughput            : 114.3 FPS

  Ultralytics breakdown (ms/image, mean)
    Preprocess          :    0.74
    Inference           :    5.92
    Postprocess         :    0.17
    ─────────────────────────────
    Total (framework)   :    6.83
────────────────────────────────────────────────────


## 6 · Save Results Summary

In [19]:
import csv

# ── Gather everything into one dict ──────────────────────────────────
summary = {
    # Run metadata
    "run_name":       RUN_NAME,
    "timestamp":      datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "best_weights":   str(best_pt.resolve()),
    "device":         DEVICE,
    # Hyperparameters
    "epochs":         EPOCHS,
    "batch":          BATCH,
    "imgsz":          IMGSZ,
    # Dataset sizes
    "train_images":   len(train_records),
    "val_images":     len(val_records),
    "test_images":    len(test_records),
    "train_boxes":    sum(len(r["labels"]) for r in train_records),
    "val_boxes":      sum(len(r["labels"]) for r in val_records),
    "test_boxes":     sum(len(r["labels"]) for r in test_records),
    # Overall metrics
    "map50":          round(float(box.map50), 4),
    "map50_95":       round(float(box.map),   4),
    "precision":      round(float(box.mp),    4),
    "recall":         round(float(box.mr),    4),
}

# Per-class AP
per_class_ap = {}
for i, cls_name in enumerate(CLASS_NAMES):
    if i < len(box.ap50):
        key = f"ap50_{cls_name}"
        val = round(float(box.ap50[i]), 4)
        summary[key]       = val
        per_class_ap[cls_name] = val

# ── 1. JSON summary (human-readable, structured) ─────────────────────
json_path = save_dir / "results_summary.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"JSON summary saved → {json_path}")

# ── 2. CSV row (easy to append across multiple runs) ─────────────────
csv_path = RESULTS_DIR / "all_runs.csv"
write_header = not csv_path.exists()
with open(csv_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary.keys()))
    if write_header:
        writer.writeheader()
    writer.writerow(summary)
print(f"CSV row appended  → {csv_path}")

# ── 3. Markdown report ───────────────────────────────────────────────
md_lines = [
    f"# Results: {RUN_NAME}",
    f"_Generated: {summary['timestamp']}_",
    "",
    "## Run Configuration",
    f"| Parameter | Value |",
    f"|-----------|-------|" ,
    f"| Model     | yolo26s.pt |",
    f"| Device    | {DEVICE} |",
    f"| Epochs    | {EPOCHS} |",
    f"| Batch     | {BATCH} |",
    f"| Image size | {IMGSZ} |",
    f"| Best weights | `{best_pt}` |",
    "",
    "## Dataset Split",
    "| Split | Images | Boxes |",
    "|-------|--------|-------|" ,
    f"| Train | {summary['train_images']} | {summary['train_boxes']} |",
    f"| Val   | {summary['val_images']}   | {summary['val_boxes']}   |",
    f"| Test  | {summary['test_images']}  | {summary['test_boxes']}  |",
    "",
    "## Test Set Metrics",
    "| Metric | Value |",
    "|--------|-------|" ,
    f"| mAP@0.50     | {summary['map50']} |",
    f"| mAP@0.50:0.95 | {summary['map50_95']} |",
    f"| Precision    | {summary['precision']} |",
    f"| Recall       | {summary['recall']} |",
    "",
    "## Per-Class AP@0.50",
    "| Class | AP@0.50 |",
    "|-------|---------|" ,
]
for cls_name, ap in per_class_ap.items():
    md_lines.append(f"| {cls_name} | {ap} |")

md_path = save_dir / "results_summary.md"
with open(md_path, "w") as f:
    f.write("\n".join(md_lines) + "\n")
print(f"Markdown report   → {md_path}")

print("\n✅ All result files saved.")
# ── Speed metrics (populated by Section 5b) ─────────────────────────
if 'speed_results' in dir():
    summary.update(speed_results)

# Add speed section to markdown report if available
if 'speed_results' in dir():
    md_lines += [
        "",
        "## Inference Speed",
        "| Metric | Value |",
        "|--------|-------|",
        f"| Latency mean (ms) | {speed_results['latency_mean_ms']:.2f} |",
        f"| Latency std (ms)  | {speed_results['latency_std_ms']:.2f} |",
        f"| Throughput (FPS)  | {speed_results['throughput_fps']:.1f} |",
        f"| Preprocess (ms)   | {speed_results['preprocess_mean_ms']:.2f} |",
        f"| Inference (ms)    | {speed_results['inference_mean_ms']:.2f} |",
        f"| Postprocess (ms)  | {speed_results['postprocess_mean_ms']:.2f} |",
    ]


JSON summary saved → /var/home/polygon/EE3703_RPC/runs/detect/yolo26s_det_grocery/results_summary.json
CSV row appended  → runs/detect/all_runs.csv
Markdown report   → /var/home/polygon/EE3703_RPC/runs/detect/yolo26s_det_grocery/results_summary.md

✅ All result files saved.


In [20]:
# ── Final recap printed inline ────────────────────────────────────────
print("\n" + "═" * 44)
print(f"  Run            : {RUN_NAME}")
print(f"  Completed at   : {summary['timestamp']}")
print(f"  Best weights   : {best_pt.name}")
print("  ─" * 22)
print(f"  mAP@0.50       : {summary['map50']}")
print(f"  mAP@0.50:0.95  : {summary['map50_95']}")
print(f"  Precision      : {summary['precision']}")
print(f"  Recall         : {summary['recall']}")
print("  ─" * 22)
for cls_name, ap in per_class_ap.items():
    print(f"  {cls_name:<22}: {ap}")
print("═" * 44)
# Speed recap
if 'speed_results' in dir():
    print('  ─' * 22)
    print(f"  Latency (ms/img)  : {speed_results['latency_mean_ms']:.2f} ± {speed_results['latency_std_ms']:.2f}")
    print(f"  Throughput        : {speed_results['throughput_fps']:.1f} FPS")
    print(f"  Preprocess (ms)   : {speed_results['preprocess_mean_ms']:.2f}")
    print(f"  Inference (ms)    : {speed_results['inference_mean_ms']:.2f}")
    print(f"  Postprocess (ms)  : {speed_results['postprocess_mean_ms']:.2f}")



════════════════════════════════════════════
  Run            : yolo26s_det_grocery
  Completed at   : 2026-03-30 17:16:00
  Best weights   : best.pt
  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─
  mAP@0.50       : 0.9533
  mAP@0.50:0.95  : 0.8515
  Precision      : 0.9144
  Recall         : 0.9184
  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─
  Bottled_Drink         : 0.9529
  Canned                : 0.9867
  Fresh_Produce         : 0.9563
  Packaged_Food         : 0.9174
════════════════════════════════════════════
  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─  ─
  Latency (ms/img)  : 8.75 ± 0.39
  Throughput        : 114.3 FPS
  Preprocess (ms)   : 0.74
  Inference (ms)    : 5.92
  Postprocess (ms)  : 0.17
